# **Report**

## **Introduction**
This report will explore **Deterministic Finite Automata (DFA)**, an important concept in theoretical computer science and mathematics. A DFA is a finite state machine that takes in string of symbols and accepts or rejects the string based on the sequence of states. DFAs have many practical applications, including recognising formal languages, malware detection, text processing, user interface and compilers. The theoretical foundations were laid by the work of Alan Turing in the 1930s, creating the Turing machine, which helped in the development of finite state machines that became essential for early computer engineers.

We will specifically explore how to identify a **synchronising DFA**, a special type of DFA that contains a reset word in its alphabet. The Cerny Conjecture states that $(n-1)^2$ is the largest length of a reset word for a DFA with n states. The conjecture made by Jan Cerny in 1969 still remains an unsolved problem in Computer science. We will discover that synchronizing DFAs correspond to a transformation semigroup, a set of functions that is closed under composition and contains a constant map (if synchronizing). One of the methods used involves creating a pair digraph for a DFA to identify if the given DFA is sychronizing. This involves finding a transformation that sends any pair of states to the same state.

We will also be exploring **isomorphisms** between two DFAs that have various levels of strictness, determining if synchronizing is an invariant property. Specifically, we will analyse strict, weak and semi isomorphisms, which allows us to compare DFAs and evaluate their properties.

## **Implementation**

#### **Strict Isomorphism**

We aim to find if two given DFAs are stricly isomorphic. To do this, we will test for if there exists an isomorhism between the DFAs.

**The general idea:**

--Define a function is_strict_isomorphism to test if A1 and A2 are strictly isomorphic

*To be strictly isomorphic*

--Need a bijection from $Q_1$ to $Q_2$, so must have same cardinalities ($|Q_1|=|Q_2|$), return False if not

--Need $f(F_1)=F_2$, since $f$ is bijection, must have $|F_1|=|F_2|$, return False if not

Now we can construct the mapping $f$ using backtrack search

**Note:** that we assumed that $Q={0,1,...,n-1}$ for some natural number $n$, with $|Q_1|=|Q_2|$ if $f$ is an injection, it is a bijection

--Create a list to track the used Q2 states, this insures injectivity of $f$

--Want $f$ to map $(q_s)_1$ to $(q_s)_2$, set $f((q_s)_1)=(q_s)_2$, mark $(q_s)_2)$ as used

--Define the recursive backtracking function, $dive(v)$, on a vertex $v$

--$dive(v)$ does:

----check if every state is mapped, if that happens, verify isomorphism conditions on $f$, returning True if passes all

----if not, then if $v$ already mapped then move to next, until an unmapped vertex is found

----using a loop, try every *unused* state $x$ of $Q_2$ as image of $v$ under $f$, verifing the isomorphism conditions hold, if violated, break out and try the next $x$ 

----do these two lines: 1. 'if $dive(v+1)$: return True', and after that 2. revert the changes i.e. $f[v]$ to $-1$ and set "usedness" of $x$ back to unusued (this line will only be done if False is returned)

----these two lines are crucial, since $dive(v+1)$ will eventually reach line 1, which procedes with $dive(v+2)$ and so on. Finally $dive(n)$ will be reached, this is the 'every(last) vertex is mapped' case, now dive verifies the isomorphism conditions and either return a True or a False

----if True is returned we found a solution, so propagate the True upwards so that $dive(v)$ returns True

----if False is returned, this is a dead end, all possible constructions from $f[v]=x$ has failed, hence does line 2 (which can be interpreted as going one step back), and explores other possiblities. This might succeed or have to go further back.

----lastly $dive(v)$ returns False. This will only be triggered if all possible constructions from $v$ has failed, hence $v$ is a dead end, so need to move back, to $v-1$.

Now out of dive

--let is_strict_isomorphism return $dive(0)$

--this will either fully construct $f$ and return a True, or will exhaust all possible constuctions from $0$ and finally having $dive(0)$ returning a False

**Discussion about exsistance of a bijection**

Since being strictly isomorphic means 'there exists an isomorphism between the DFAs' and we constucted a specific one. If the $f$ constructed fails to be an isomorphism, could there still exist an isomorphism $g$, making the DFAs isomorphic? 

To explain this, note that backtracking search exhausts every possible bijection, so is_strict_isomorphism searches *every* bijection that maps $(q_s)_1$ to $(q_s)_2$. It tries every possible mapping to an unmapped state, with this, if an isomorphism existis, it *will* find one. So returning a False really means there does not exist such an isomorphism.



## **Code**

#### **Strict Isomorphism**

In [ ]:
def is_strict_isomorphism(A1,A2):
    """
    Tests if two DFAs A1=(Q1,Σ,τ1,qs1,F1) and A2=(Q2,Σ,τ2,qs2,F2) are strictly isomorphic.
    Assumes they have the common alphabet Σ.
    Assumes every state in Q is reachable from qs.
    Uses backtrack search.

    Parameters:
    - A1, A2: Two DFAs
    Arguments of A1, A2:
    - Q1, Q2: some finite set of states
    - Σ: some finite alphabet (a set)
    - τ1, τ2: transition function τi:Qi×Σ→Qi for i=1,2 (a dict)
    - qs1,qs2: start state
    - F1, F2: set of accepting or final states

    Returns: 
    Boolean True or False
    """
    # Store the arguments of the DFAs in variables(if not already in this form)
    Q1,Σ,τ1,qs1,F1 = A1
    Q2,Σ,τ2,qs2,F2 = A2
    # Test the cardinalities of the Qs and the Fs are equal
    if len(Q1)!=len(Q2):
        return False
    if len(F1) != len(F2):
        return False
    # Build the mapping, since Q1 = Q2 = {0,1,...,n-1}, injective implies bijective.
    n = len(Q1)
    f = [-1]*n   # for q in Q1, f[q]=f(q), the image of q in Q2, -1 means not mapped yet.
    used = [False] * n   # Track which states of Q2 are already used, ensures f injective, hence bijective
    # Map qs1 to qs2 i.e. f(qs1)=qs2
    f[qs1] = qs2
    used[qs2] = True
    # Define the recursive backtracking function
    def dive(v):   # v is a vertex/state, will run dive from v=0
        if v == n:   # If we have mapped all vertices, verify the isomorphism conditions are satisfied
            for q in Q1:
                if (q in F1)!=(f[q] in F2):
                    return False
                for a in Σ:
                    if τ2[(f[q], a)] != f[τ1[(q, a)]]:
                        return False
            return True   # f satisfies the conditions and is a bijection, i.e. isomorphism, so A1 A2 stricly isomorphic.
        if f[v] != -1:
            return dive(v+1)     # If the current vertex is already mapped, skip to the next
        # Try every unused state in Q2, x, as a image for v (i.e. to see if we can set f(v)=x)
        for x in Q2:
            if used[x]:   
                continue   # x has been used, get a new x
            # Check f(F1)=F2, if false can jump right into next loop, saving time 
            if (v in F1) != (x in F2):
                continue
            # Check consistency with already mapped vertices u in Q1
            consistent = True
            for u in Q1:
                if f[u] == -1:
                    continue   # Find those mapped vertices, skipping the unmapped
                for a in Σ:
                    if τ1[(u,a)] == v:   # If from u we go to v on some symbol,
                        if τ2[(f[u],a)] != x:
                            consistent = False   # x is not f(v), since f(v)=f(τ1(u,a))=τ2(f(u),a)
                            break
                    if τ1[(v,a)] == u:   # If from v we go to u on some symbol,
                        if τ2[(x,a)] != f[u]:   
                            consistent = False   # if x=f(v), then f(u)=f(τ1(v,a))=τ2(f(v),a)=τ2(x,a)
                            break
                if not consistent:   # If consistent is False, break out immediately, if True, loop on with next u.
                    break
            if not consistent:   # If consistent is False, this x is inconsistent, go to next x
                continue
            # If reaches here, consistent is True throughout, so try to map v to x, then run dive reapeatedly.
            f[v] = x
            used[x] = True
            if dive(v+1):
                return True   # To reach here, must reached v=n and f[v:n] satisfies the isomorphism conditions.
            f[v]=-1   # Reaching here implies that something went wrong in the dive(v+1), dive(v+2),...loops
            used[x] = False    # so this x does not work, revert all succeeding vs and xs and try with another x
        # Reaching here means that no x worked, so v cant have a image satisfying the isomorphism conditions.
        return False   # This will show that v is a 'dead end' after every possible extenstion, so having to go to higher up, to v-1
    return dive(0)   # This will either fully construct f and returning a True, or, after trying every possible path, return to dive(0) and return a False

## **Demonstration of Code**

## **Description of Results**

#### **To what extent do Isomorphisms preverve synchronizing properties of DFAs (if any)**

**Strict**

Strict isomorphism will fully preserve synchronization properties of DFAs (if at all). This is since two strictly isomorphic DFAs can be thought as renaming the states, with structures identical, so if a reset word exists in one, it also exists in the other, but probably  resets to a different state.

To explicitly show this, let $A_1=(Q_1,Σ,τ_1,(q_s)_1,F_1)$ and $A_2=(Q_2,Σ,τ_2,(q_s)_2,F_2)$ be strictly isomorphic, $f$ be an isomorphism between them. For $x∈Q_1$ define $t_1(x,w)=y$ to be "transition function of words", i.e using a word $w$ on a state $x$ gives state $y$. With $w=w_1w_2...w_n$ where each $w_i∈Σ$ for finite natural $i$, $t_1(x,w)$ is basically $τ_1(...(τ_1(τ_1(x,w_1),w_2),...,),w_n)$. Using induction on length of $w$, $t_2(f(q),w)=f(t_1(q,w))$ for all $q∈Q_1$ and all word $w$ over $Σ$ can be shown to hold, but too lengthy to write here.

If $A_1$ is synchronizing, there exists a reset word $w$ over $Σ$, and resets to a state $z∈Q_1$, i.e. $t_1(q,w)=z$ for all $q∈Q_1$. Then $f(z)=f(t_1(q,w))=t_2(f(q),w)$ for all $f(q)=q_2∈Q_2$ (by surjectivity of $f$), so $w$ is a reset word and $f(z)∈Q_2$ is the state that is resetted to, so $A_2$ is synchronizing

**Weak**

**Semi**

## **Analysis of Results**

## **Who did what**
<br>
We decided to delegate the project by assigning Finlay, James and Somhairle to Part 1 and Elena, Ella and Fei to Part 2. Within the part 2 group, Fei was responsible for the strict isomorphism, Elena for the semi isomorphism and Ella for the week isomorphism. We each took responsibility in explaining and interpreting the results of our code, then collaborated in making the report and adding in the visualisations and ... for part 3.

## **Decription of the ways which we have gone beyond the project**

## **AI Use**